# OSSL to ring trial transfer

> Test whether OSSL-trained MIR models transfer to the instruments of the soil spectroscopy ring trial

::: {.callout-warning title="Work in progress"}
Pretraining and the zero-shot experiment have run. The other experiments below are still to come.
:::

This project tests whether CNN models of soil mid-infrared (MIR) spectra, trained on the [Open Soil Spectral Library (OSSL)](https://explorer.soilspectroscopy.org/), transfer to other spectrometers. It uses the ring trial of [Safanelli et al. (2023)](https://doi.org/10.1016/j.geoderma.2023.116724), in which 20 laboratories scanned the same 70 soils, each with its own instrument and procedure. A Shimadzu instrument in Japan (`Hokudai`) also scanned 59 of the soils. The targets are organic carbon, clay, pH and exchangeable potassium (Kex).

The ring trial keeps the soils fixed and changes only the instrument. 60 of the soils come from KSSL, the largest part of OSSL. One of the instruments, the IAEA Thermo Nicolet, also measured the Fukushima soils of [`ossl-fuku-transfer`](https://franckalbinet.github.io/ossl-fuku-transfer/). There the OSSL Kex model failed on the Fukushima spectra, and it is not settled whether the instrument or the soils caused it. If the model works on the IAEA spectra of these soils, the instrument is not the main cause.

The audience is soil scientists who transfer spectral models between instruments, and the readers of `ossl-fuku-transfer`.

## Design

- Data: soilspecdata's `load_ringtrial` returns one `Spectra` per instrument, with the wet chemistry of the 70 soils.
- Pretraining: OSSL v1.2 without 38 KSSL samples that are ring-trial soils. Their OSSL spectra correlate with the KSSL instrument's ring-trial spectra at 0.99999 or more, and their clay, pH and Kex values are identical. The remaining 22 KSSL soils and the 10 NAPT soils have no match in OSSL.
- Model: soilspecdl's default `SpectralCNN`, the same CNN as in `ossl-fuku-transfer`, so the results of the two projects compare directly.
- Split: on each instrument, grouped k-fold cross-validation over all 70 soils, repeated with different fold assignments. Near-duplicate soils, such as RT_04, RT_05, RT_07, RT_13 and RT_15, stay in one fold. The paper's Kennard-Stone split (50 calibration and 20 test soils) is not used: it is optimistic, and its test set has only 20 soils.
- Scores: Lin's CCC, the paper's main metric, and R². Every method uses the same folds, so methods are compared soil by soil, with paired tests.
- Experiments, in order:
    1. Zero-shot: the OSSL models on every instrument, against the paper's Cubist, MBL and PLSR transfer results.
    2. Few-shot: fine-tuning, and a ridge regression on frozen block-4 features, against PLSR trained only on the ring-trial soils of the same instrument, with growing numbers of training soils.
    3. The gains per instrument, against the paper's instrument clusters and spectral dissimilarity.

## Notebooks

- [`00_data`](00_data.ipynb): loads the spectra and the instrument table, groups the near-duplicate soils, and lists the 38 ring-trial soils that are in OSSL.

- [`01_pretrain`](01_pretrain.ipynb): trains one CNN each for clay, pH and K on OSSL, without those 38 soils, on SNV spectra, with the training settings of `ossl-fuku-transfer` but 80 epochs instead of 20.
- [`02_zero_shot`](02_zero_shot.ipynb): scores the three models on all 21 ring-trial instruments, with no adaptation.

## Set-up

The project uses the soilspec packages ([soilspecdl](https://github.com/franckalbinet/soilspecdl), [soilspecdata](https://github.com/franckalbinet/soilspecdata) and [soilspectfm](https://github.com/franckalbinet/soilspectfm)) as editable installs from sibling folders:

```
soilspec/
  ossl-rt-transfer/
  soilspecdl/
  soilspecdata/
  soilspectfm/
```

Clone the four repositories into one folder, then, in `ossl-rt-transfer/`:

```sh
uv sync --extra dev
```

The ring-trial data are not public. This repository does not contain them. With access to them, put them in `data/ringtrial/` in the workspace folder, beside the project folders. `nbdev-test` skips the cells that read them. Run `nbdev-test --flags rtdata` to include them.